# 01.06 Groq — 초저지연 추론

Groq는 자체 **LPU**(Language Processing Unit) 하드웨어로 오픈 가중치 모델(Llama, Mixtral, Qwen, GPT-OSS 등)을 토큰 수천/초 급으로 돌린다. OpenAI 호환 포맷이지만 `langchain-groq`가 공식 래퍼를 제공한다.

## 학습 목표

- `ChatGroq(model="llama-3.3-70b-versatile")` 기본 사용
- `reasoning_format` / `reasoning_effort` 로 reasoning 모델(deepseek-r1-distill 등) 제어
- tool calling 지원 모델 확인

## 언제 쓰나

- **UX가 지연에 민감**한 리얼타임 채팅 / 음성 에이전트
- 오픈 가중치 모델을 관리형으로 쓰되 로컬 GPU는 두기 싫을 때
- 대용량 배치가 아닌 대화 지향 워크로드

## 01.06.1 환경 설정

In [ ]:
# !pip install -U langchain langchain-groq
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY 필요"

## 01.06.2 기본 사용

Groq의 대표 모델 ID(2026-04 기준): `llama-3.3-70b-versatile`, `llama-3.1-8b-instant`, `mixtral-8x7b-32768`, `qwen-2.5-32b`, `deepseek-r1-distill-llama-70b`.

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
msg = model.invoke("Groq LPU의 핵심 장점을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.06.3 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model("groq:llama-3.1-8b-instant")
print(m.invoke("핑").content[:80])

## 01.06.4 Tool calling

Groq에서 tool calling을 지원하는 모델: Llama 3.1/3.3 전체, Mixtral, Qwen-2.5.

In [ ]:
from langchain_core.tools import tool

@tool
def stock(ticker: str) -> float:
    """티커의 종가를 반환한다."""
    return {"NVDA": 920.5}.get(ticker.upper(), 0.0)

bound = model.bind_tools([stock])
out = bound.invoke("NVDA 종가 알려줘")
print("tool_calls:", out.tool_calls)

## 01.06.5 Reasoning 모델 — `reasoning_format`, `reasoning_effort`

DeepSeek-R1-Distill 같은 reasoning 모델은 `<think>…</think>` 블록을 출력할 수 있다. Groq는 이를 파싱해 별도 필드로 분리해 주는 파라미터를 제공한다.

- `reasoning_format="parsed"` — `additional_kwargs["reasoning_content"]`로 추출
- `reasoning_format="hidden"` — 드롭
- `reasoning_format="raw"` — 원문 그대로
- `reasoning_effort` — `low | medium | high`(지원 모델만)

In [ ]:
reasoner = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    reasoning_format="parsed",
    reasoning_effort="medium",
)
r = reasoner.invoke("3자리 수 144의 제곱근을 단계별로 구해줘.")
print("content      :", r.content[:200])
print("reasoning len:", len((r.additional_kwargs or {}).get("reasoning_content", "")))

## 01.06.6 Structured output

In [ ]:
from pydantic import BaseModel

class Ticker(BaseModel):
    symbol: str
    close: float

structured = model.with_structured_output(Ticker)
print(structured.invoke("TSLA 종가는 250.1이야. 구조화해줘."))

## 체크리스트

- [ ] 모델 ID가 현재 Groq 카탈로그에 있는지 확인했다 (모델 수명이 짧다)
- [ ] reasoning 모델에는 `reasoning_format`을 명시했다
- [ ] tool calling 지원 모델만 `bind_tools` 경로에 사용했다

## 다음

- `04_ollama.ipynb` — 같은 오픈 가중치 모델의 로컬 경로
- `09_routers_and_aggregators.ipynb` — 여러 공급자를 한 코드베이스에서

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/groq
- Groq 모델 카탈로그: https://console.groq.com/docs/models